In [1]:
import pandas as pd

fondos_mapeados = pd.read_csv('data/funds_prices.csv')

fondos_mapeados

/var/folders/s9/mpd6xx3x1173hcqlz6vh5s_80000gn/T/ipykernel_33230/48335682.py:3: DtypeWarning: Columns (165,167,207,303,304,305,306,307,308,309,310,311,312,313,314,315,316,501,505,548) have mixed types. Specify dtype option on import or set low_memory=False.
  fondos_mapeados = pd.read_csv('data/funds_prices.csv')


,Unnamed: 0,Dates,0JKT LN Equity,AAXJ US Equity,ABCHNSA LX EQUITY,BGFPABA LN EQUITY,CPXJ LN EQUITY,CAFINIV LX EQUITY,FFASAIU LX EQUITY,FFEMASI LX EQUITY,...,CFIETFCC CC Equity,MBIDTANV CI Equity,MORCLPR CI EQUITY,CFIMRCLP CI EQUITY,SECRNOM CI EQUITY,SECR1NV CI EQUITY,FILRDCM CI EQUITY,FALCDCW CI EQUITY,IMTSCLD CI EQUITY,FYDCLAC CI EQUITY
0,0,2007-01-01,NaN,NaN,NaN,4.13102,NaN,124.47,NaN,NaN,...,NaN,NaN,NaN,5.420810e+03,NaN,NaN,NaN,NaN,NaN,NaN
1,1,2007-01-02,NaN,NaN,NaN,4.17635,NaN,125.69,NaN,NaN,...,NaN,NaN,NaN,5.459870e+03,NaN,NaN,NaN,NaN,NaN,NaN
2,2,2007-01-03,NaN,NaN,NaN,4.11882,NaN,126.26,NaN,NaN,...,NaN,NaN,NaN,5.473550e+03,NaN,NaN,NaN,NaN,NaN,NaN
3,3,2007-01-04,NaN,NaN,NaN,4.03831,NaN,125.45,NaN,NaN,...,NaN,NaN,NaN,5.503620e+03,NaN,NaN,NaN,NaN,NaN,NaN
4,4,2007-01-05,NaN,NaN,NaN,4.05409,NaN,124.76,NaN,NaN,...,NaN,NaN,NaN,5.506130e+03,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4956,4956,2025-12-30,33.03644,128.2802,140.00,21.17484,220.75,506.07,16.62,29.22,...,1523.160,17393.3932,20379.16,2.765698e+08,238010.3987,1904.7218,11742.66,1442.714428,1876.0906,1502.3213
4957,4957,2025-12-31,33.03644,127.8545,139.59,21.28106,219.29,509.63,16.57,29.18,...,1523.160,17395.8722,20383.28,2.766311e+08,238010.3987,1904.7218,11744.42,1442.940869,1876.3625,1502.5413
4958,4958,2026-01-01,33.03644,127.8545,139.59,21.28106,219.29,509.63,16.57,29.18,...,1523.160,17397.2962,20386.70,2.766775e+08,238010.3987,1904.7218,11746.28,1443.138425,1876.6366,1502.7609
4959,4959,2026-01-02,33.03644,131.8500,139.59,21.60812,220.43,512.41,16.60,29.80,...,1525.530,17397.2962,20386.70,2.766775e+08,238010.3987,1904.7218,11747.80,1443.138425,1876.6366,1502.5780


In [2]:
diccionario = pd.read_csv('data/funds_dictionary.csv')
diccionario

,Unnamed: 0,Ticker,Asset Class Sistema,Asset Class,Geography / Subclass,Indice,Market
0,0,0JKT LN EQUITY,RENTA VARIABLE,RENTA VARIABLE,Asia ex-Japan,ISHARES TRUST ISHARES MSCI INDIA ETF,India
1,1,AAXJ US EQUITY,RENTA VARIABLE,RENTA VARIABLE,Asia ex-Japan,ISHARES MSCI ALL COUNTRY ASIA ES Japan,General
2,2,ABCHNSA LX EQUITY,RENTA VARIABLE,RENTA VARIABLE,Asia ex-Japan,AB SICAV I CHINA A SHARES EQUITY PORTFOLIO,China
3,3,BGFPABA LN EQUITY,RENTA VARIABLE,RENTA VARIABLE,Asia ex-Japan,BAILLIE GIFFORD OVERSEAS GROWTH FUNDS ICVC BAI...,General
4,4,CPXJ LN EQUITY,RENTA VARIABLE,RENTA VARIABLE,Asia ex-Japan,ISHARES MSCI PACIFIC EX Japan UCITS ETF,General
...,...,...,...,...,...,...,...
618,618,SECR1NV CI EQUITY,RENTA FIJA,RENTA FIJA,Chile,FONDO DE INVERSION SECURITY RENTA FIJA NACIONAL,Fund
619,619,FILRDCM CI EQUITY,RENTA FIJA,RENTA FIJA,Chile,FONDO DE INVERSION LARRAINVIAL DEUDA CHILE,Fund
620,620,FALCDCW CI EQUITY,RENTA FIJA,RENTA FIJA,Chile,FALCOM DEUDA CORPORATIVA CHILE,Fund
621,621,IMTSCLD CI EQUITY,RENTA FIJA,RENTA FIJA,Chile,FONDO DE INVERSION CREDICORP CAPITAL SPREAD CO...,Fund


In [3]:
import re
import pandas as pd

# --- 1) Cargar data ---
precios = pd.read_csv('data/funds_prices.csv')
dicc   = pd.read_excel('/Users/matias/Desktop/Proyectos/ranking-fondos/dict_temp_full_portfolio.xlsx', sheet_name='Hoja1')

# Garantiza que la columna de fechas se llame 'Dates' y sea datetime
if 'Dates' in precios.columns:
    precios['Dates'] = pd.to_datetime(precios['Dates'])
else:
    # Si tu CSV trae otro nombre, cámbialo aquí
    raise ValueError("No se encuentra columna 'Dates' en funds_prices.csv")

# --- 2) Normalizador de tickers (equivale 'Equity'/'EQUITY', colapsa espacios, mayúsculas) ---
def norm_ticker(x: str) -> str:
    if pd.isna(x):
        return x
    s = str(x).strip()
    s = re.sub(r'\s+', ' ', s)            # colapsa múltiples espacios
    s = s.upper()                          # todo a MAYÚSCULAS
    s = re.sub(r'\s+EQUITY$', ' EQUITY', s)  # sufijo EQUITY estandar
    return s

# --- 3) Estandarizar tickers del diccionario ---
dicc = dicc.copy()
if 'Ticker' not in dicc.columns:
    raise ValueError("El diccionario debe tener columna 'Ticker'")
dicc['Ticker_norm'] = dicc['Ticker'].apply(norm_ticker)

# --- 4) Estandarizar columnas del dataframe de precios (excepto 'Dates') ---
price_cols = [c for c in precios.columns if c != 'Dates']
map_cols = {c: norm_ticker(c) for c in price_cols}
precios_std = precios.rename(columns=map_cols)

# Conjuntos útiles
tickers_precios = set(map_cols.values())
tickers_dicc    = set(dicc['Ticker_norm'])

# --- 5) Detectar y reportar no mapeados (para que los corrijas luego si hace falta) ---
no_mapeados = sorted(tickers_precios - tickers_dicc)
# (Opcional) Guarda un CSV para revisarlos:
pd.DataFrame({'Ticker_no_mapeado': no_mapeados}).to_csv('data/tickers_no_mapeados.csv', index=False)

# --- 6) Seleccionar CHILE ---
# Asumimos que 'Geografia' indica el país/region y que Chile aparece como 'Chile'.
# Además, por si tienes acciones locales por ISIN, los de Chile parten con 'CL'.
dicc['Geografia/Subclass'] = dicc['Geografia/Subclass'].fillna('')

mask_chile = (dicc['Geografia/Subclass'].str.strip().str.casefold() == 'chile') | dicc['Ticker_norm'].str.startswith('CL')
dicc_chile = dicc.loc[mask_chile, ['Ticker_norm']].drop_duplicates()

# Tickers de Chile que SÍ están en el dataframe de precios
tickers_chile = sorted(t for t in dicc_chile['Ticker_norm'] if t in tickers_precios)

# --- 7) Separar dataframes: CHILE vs resto ---
cols_chile = ['Dates'] + tickers_chile
df_chile = precios_std.loc[:, [c for c in cols_chile if c in precios_std.columns]].copy()

tickers_resto = sorted(tickers_precios - set(tickers_chile))
cols_resto = ['Dates'] + tickers_resto
df_resto = precios_std.loc[:, [c for c in cols_resto if c in precios_std.columns]].copy()

# --- 8) (Opcional) Guardar resultados para tu pipeline de moneda a futuro ---
df_chile.to_csv('data/funds_prices_chile.csv', index=False)
df_resto.to_csv('data/funds_prices_exchile.csv', index=False)

# --- 9) Resumen rápido por consola ---
print(f"Tickers totales en precios: {len(tickers_precios)}")
print(f"Tickers mapeados en dicc:   {len(tickers_dicc)}")
print(f"Tickers Chile detectados:   {len(tickers_chile)}")
print(f"No mapeados (revisar):     {len(no_mapeados)} -> data/tickers_no_mapeados.csv")


/var/folders/s9/mpd6xx3x1173hcqlz6vh5s_80000gn/T/ipykernel_33230/781306692.py:5: DtypeWarning: Columns (165,167,207,303,304,305,306,307,308,309,310,311,312,313,314,315,316,501,505,548) have mixed types. Specify dtype option on import or set low_memory=False.
  precios = pd.read_csv('data/funds_prices.csv')


KeyError: 'Geografia/Subclass'

In [ ]:
# --- 10) Exportar listas de tickers a Excel (Chile / Internacional) ---
from pathlib import Path

# Mapa inverso para recuperar el nombre original como venía en funds_prices.csv
inv_map_cols = {v: k for k, v in map_cols.items()}

# DataFrames de listas
df_list_chile = pd.DataFrame({
    "Ticker_norm": tickers_chile,
    "Ticker_original": [inv_map_cols.get(t, t) for t in tickers_chile]
})

tickers_internac = sorted(tickers_precios - set(tickers_chile))
df_list_internac = pd.DataFrame({
    "Ticker_norm": tickers_internac,
    "Ticker_original": [inv_map_cols.get(t, t) for t in tickers_internac]
})

# (Opcional) también dejar los no mapeados para revisión
df_no_mapeados = pd.DataFrame({"Ticker_no_mapeado": no_mapeados})

# Ruta de salida
out_xlsx = Path("data") / "listas_tickers_por_region.xlsx"

with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    df_list_chile.to_excel(writer, sheet_name="Chile", index=False)
    df_list_internac.to_excel(writer, sheet_name="Internacional", index=False)
    df_no_mapeados.to_excel(writer, sheet_name="No_mapeados", index=False)

print(f"✔️ Archivo creado: {out_xlsx.resolve()}")
print(f"Hojas: Chile ({len(df_list_chile)}), Internacional ({len(df_list_internac)}), No_mapeados ({len(df_no_mapeados)})")


✔️ Archivo creado: /Users/matias/Desktop/Proyectos/ranking-fondos/data/listas_tickers_por_region.xlsx
Hojas: Chile (20), Internacional (318), No_mapeados (8)
